In [0]:
from pyspark.sql import functions as F
lines = spark.table("claims.silver.service_line")
hdr   = spark.table("claims.silver.claim_header")

proc_perf = (lines.groupBy("hcpcs_code").agg(
        F.count("*").alias("line_count"),
        F.sum("line_allowed").alias("total_allowed"),
        F.sum("line_paid").alias("total_paid"),
        F.round(F.avg(F.col("is_zero_pay").cast("int")) * 100, 2).alias("zero_pay_pct"),
        F.sum("payment_variance").alias("total_variance"),
        F.round(F.avg("patient_responsibility"), 2).alias("avg_patient_resp"))
    .filter("line_count >= 100")
    .withColumn("payment_rate",
        F.round(100 * F.col("total_paid") / F.col("total_allowed"), 2)))

proc_perf.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("claims.gold.agg_procedure_payment")

prov_perf = (hdr.groupBy("provider_id", "claim_type").agg(
        F.count("*").alias("claims"),
        F.sum("paid_amount").alias("total_paid"),
        F.round(F.avg("paid_amount"), 2).alias("avg_paid"),
        F.round(F.avg("service_days"), 1).alias("avg_length_of_stay"))
    .filter("claims >= 25"))

prov_perf.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("claims.gold.agg_provider_payment")

monthly = (hdr.groupBy("service_month", "claim_type").agg(
        F.count("*").alias("claims"),
        F.sum("paid_amount").alias("total_paid"))
    .orderBy("service_month"))

monthly.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("claims.gold.agg_monthly_claims")